In [1]:
%cd ../..

/path/to/repo/my-coles-gnn-experimetns/scenario_gender


/path/to/repo/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
PRETRAINED_MODELS_ROOT = 'models/pretrained_gnn_models'

In [3]:
from collections import defaultdict
import os

CHECKPOINTS_DIR = "../../checkpoints"

dir_to_epoches = {
    "wl-0.5_gnn-GAT_res-True_wd-0": [15, 50, 75, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.1": [25, 50, 75, 100, 150, 200],
    "wl-0.5_gnn-GAT_res-True_wd-0.01": [25, 50, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-False_wd-0": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0": [75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.1": [50, 75, 100, 150, 200],
    "wl-0.5_gnn-GraphSAGE_res-True_wd-0.01": [25, 50, 75, 100, 150, 200],
}



MAX_EPOCHES=40
N_TRIPLETS_PER_ANCHOR_USER_OPTIONS=(128, 32, 8, 1)
MIN_ELEMENTS_IN_BIN_WHEN_ONE_BIN_ONLY_OPTIONS=(5,)
LOSS_ALPHA_OPTIONS=(0.5, 0.85, 0.15)

config_item_template = """  {experiment_name}:
    enabled: true
    read_params: 
      file_name: ${{hydra:runtime.cwd}}/data/{experiment_name}_embeddings.pickle
    target_options: {{}}
"""


existing_experiments = []

for pretrained_gnn_name, pretrain_epoches in dir_to_epoches.items():
    for pretrain_epoch in pretrain_epoches:
        for min_elements_in_bin_when_one_bin_only in MIN_ELEMENTS_IN_BIN_WHEN_ONE_BIN_ONLY_OPTIONS:
            for n_triplets_per_anchor_user in N_TRIPLETS_PER_ANCHOR_USER_OPTIONS:
                for loss_alpha in LOSS_ALPHA_OPTIONS:
                    
                    experiment_name = f'check__coles_bpr_with_pretrained_gnn__GNN_{pretrained_gnn_name}__pretrain_epoches_{pretrain_epoch}__loss_alpha_{loss_alpha}__triplets_per_user_{n_triplets_per_anchor_user}__bin_separation_margin_{min_elements_in_bin_when_one_bin_only}__{MAX_EPOCHES}__epoches__try_1'
                    ckpt_dir_path = os.path.join(CHECKPOINTS_DIR, experiment_name)
                    if os.path.exists(f'data/{experiment_name}_embeddings.pickle'):
                        existing_experiments.append(experiment_name)



In [4]:
for experiment in existing_experiments:
    print(config_item_template.format(experiment_name=experiment))

  check__coles_bpr_with_pretrained_gnn__GNN_wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_75__loss_alpha_0.5__triplets_per_user_128__bin_separation_margin_5__40__epoches__try_1:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/check__coles_bpr_with_pretrained_gnn__GNN_wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_75__loss_alpha_0.5__triplets_per_user_128__bin_separation_margin_5__40__epoches__try_1_embeddings.pickle
    target_options: {}

  check__coles_bpr_with_pretrained_gnn__GNN_wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_75__loss_alpha_0.85__triplets_per_user_128__bin_separation_margin_5__40__epoches__try_1:
    enabled: true
    read_params: 
      file_name: ${hydra:runtime.cwd}/data/check__coles_bpr_with_pretrained_gnn__GNN_wl-0.5_gnn-GAT_res-True_wd-0__pretrain_epoches_75__loss_alpha_0.85__triplets_per_user_128__bin_separation_margin_5__40__epoches__try_1_embeddings.pickle
    target_options: {}

  check__coles_bpr_with_pretrained_gnn__GNN_wl

In [5]:
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

In [6]:
from ptls_extension_2024_research.latex_table_creation import create_latex_table, get_metrics


/path/to/repo/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from typing import List, Dict, Any


def assert_metric_in_data_and_number(data_lst: List[Dict[str, Any]], metric_name: str):
    for el in data_lst:
        assert metric_name in el, f"Metric name {metric_name} not found in data"
        assert isinstance(el[metric_name], (int, float)), f"Metric {metric_name} should be a number. Found {type(el[metric_name])}"


def sort_by_col(data_lst: List[Dict[str, Any]], metric_name: str):
    assert_metric_in_data_and_number(data_lst, metric_name)
    data_lst = sorted(data_lst, key=lambda x: x[metric_name], reverse=True)
    return data_lst


def bolden_top_k(data_lst: List[Dict[str, Any]], k, metric_names: List[str]):
    for metric_name in metric_names:
        assert_metric_in_data_and_number(data_lst, metric_name)

    for metric_name in metric_names:
        sorted_with_idxs = sorted(enumerate(data_lst), key=lambda x: x[1][metric_name], reverse=True)
        for i, data in sorted_with_idxs[:k]:
            data_lst[i][metric_name] = "\\textbf{" + str(data[metric_name]) + "}" 
    return data_lst


def get_idxs_where_all_metrics_superpass(data_lst, metric_name_to_limit_val: Dict[str, float]):
    idxs = []
    for i, data in enumerate(data_lst):
        superpass = True
        for metric_name, limit_val in metric_name_to_limit_val.items():
            if data[metric_name] < limit_val:
                superpass = False
                break
        if superpass:
            idxs.append(i)
    return idxs

def prefix_map_from_idx_lst(idx_lst, prefix):
    return {i: prefix for i in idx_lst}

In [11]:
with open("results/scenario_gender_bpr_with_pretrained_gnn.txt") as f:
    report_file_content = f.read()

In [12]:
COLES_METRICS = {r"\textbf{AUC test}": 0.877, r"\textbf{ACC test}": 0.793}

In [13]:
hyperparams_to_getters = {
    r"\textbf{Triplets per user}": lambda config: config['pl_module']['loss']['loss2']['triplet_selector']['num_triplets_per_anchor_user'],
    r"\textbf{Loss alpha}": lambda config: config['pl_module']['loss']['alpha'],
    r"\textbf{Min users in separated single bin}": lambda config: config['pl_module']['loss']['loss2']['triplet_selector']['bin_separation_strategy']['min_elements_in_bin']
}

In [19]:
import re

hyperparams_to_getters = {
    r"\textbf{GNN}": lambda experiment_name: re.search(r'gnn-(GraphSAGE|GAT)', experiment_name).group(1),
    r"\textbf{Has residual connection}": lambda experiment_name: True if re.search(r'res-(True|False)', experiment_name).group(1) == "True" else False,
    r"\textbf{Triplets per user}": lambda experiment_name: int(re.search(r'triplets_per_user_(\d+)', experiment_name).group(1)),
    r"\textbf{Loss alpha}": lambda experiment_name: float(re.search(r'loss_alpha_(\d+\.\d+)', experiment_name).group(1)),
    r"\textbf{Min users in separated single bin}": lambda experiment_name: int(re.search(r'bin_separation_margin_(\d+)', experiment_name).group(1)),
    r"\textbf{GNN Loss alpha}": lambda experiment_name: float(re.search(r'wl-(\d+\.\d+)', experiment_name).group(1)),
    r"\textbf{Pretrain epoches}": lambda experiment_name: int(re.search(r'pretrain_epoches_(\d+)', experiment_name).group(1)),
    r"\textbf{Train epoches}": lambda experiment_name: int(re.search(r'(\d+)__epoches', experiment_name).group(1))
}


In [20]:
experiment_dicts_list = []

for experiment_name in existing_experiments:
    hyperparams = {k: v(experiment_name) for k, v in hyperparams_to_getters.items()}
    metrics = get_metrics(report_file_content, experiment_name, {'auroc': r"\textbf{AUC test}", 'accuracy': r"\textbf{ACC test}"})
    experiment_dicts_list.append({**hyperparams, **metrics})

In [21]:
WEIGHTS_TYPE = 'type 2'


experiment_dicts_list = sort_by_col(experiment_dicts_list, r"\textbf{AUC test}")
prefix_map = prefix_map_from_idx_lst(get_idxs_where_all_metrics_superpass(experiment_dicts_list, COLES_METRICS), "\n\\rowcolor{gray!50}\n")
experiment_dicts_list = bolden_top_k(experiment_dicts_list, 3, [r"\textbf{AUC test}", r"\textbf{ACC test}"])




EXPERIMENT_NAME = r'\makecell{Coles + pretrained gnn with bpr}'

experiment_data = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: experiment_dicts_list
    }
}



In [22]:
hyperparameters = hyperparameter_header_strs = list(experiment_dicts_list[0].keys())

caption = r"Comparison of different setups where coles had item\_id (mcc_code) embedding layer initialized with pretrained gnn embeddings and was trained with contrastive loss alongside with bpr loss. Bpr loss chooses positive and negative users according to similarity of user embeddings extracted from adjacency matrix"

row_prefix_dict = {
    EXPERIMENT_NAME: {
        WEIGHTS_TYPE: prefix_map
    }
}

latex_table = create_latex_table(experiment_data, hyperparameters, hyperparameter_header_strs, caption, row_prefix_dict)

print(latex_table)

\begin{table*}[h!]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|c|c|}
\hline
\textbf{Model} & \textbf{Type of weights} & \textbf{GNN} & \textbf{Has residual connection} & \textbf{Triplets per user} & \textbf{Loss alpha} & \textbf{Min users in separated single bin} & \textbf{GNN Loss alpha} & \textbf{Pretrain epoches} & \textbf{Train epoches} & \textbf{AUC test} & \textbf{ACC test}\\
\hline

\rowcolor{gray!50}
\multirow{20}{*}{\makecell{Coles only with bpr}} &\multirow{20}{*}{type 2} & GAT & True & 8 & 0.15 & 5 & 0.5 & 75 & 40 & \textbf{0.882} & \textbf{0.799} \\ \cline{3-12} 

\rowcolor{gray!50}
& &GraphSAGE & True & 128 & 0.15 & 5 & 0.5 & 100 & 40 & \textbf{0.881} & \textbf{0.799} \\ \cline{3-12} 

\rowcolor{gray!50}
& &GraphSAGE & True & 8 & 0.5 & 5 & 0.5 & 100 & 40 & \textbf{0.881} & 0.798 \\ \cline{3-12} 

\rowcolor{gray!50}
& &GAT & True & 128 & 0.85 & 5 & 0.5 & 75 & 40 & 0.88 & 0.795 \\ \cline{3-12} 
& &GraphSAGE & True & 128 & 0.85 & 5 & 0.5 & 100 